# HF Inference Client LLM Comparison (Tweet Sentiment, Social Bias Frames, AG News)

This notebook runs a simple, reproducible experiment across four LLMs using Hugging Face Inference Client:
- Qwen/Qwen2.5-1.5B-Instruct
- Llama 8B (with fallback mapping if exact 3.2 8B endpoint is unavailable)
- openai/gpt-oss-20b
- Mixtral 8x22B

For each dataset, it samples 50 test rows (stratified when possible), predicts in batched requests (default batch size = 5), and computes metrics.

Outputs include:
- Per-model predictions for each exact sample
- all_agree and major_vote columns
- Accuracy / macro-F1 / weighted-F1
- Logprob summaries where the endpoint exposes them
- Saved artifacts under results/hf_llms_comparison/

Social Bias Frames is loaded from the SBIC v2 archive and benchmarked as a binary biased-implication task.

In [3]:
import io
import json
import os
import re
import tarfile
import time
from collections import Counter
from datetime import datetime
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd
import requests

try:
    from datasets import load_dataset
except ModuleNotFoundError as e:
    raise RuntimeError(
        "Missing package: datasets. Install once with: pip install datasets"
    ) from e

try:
    from huggingface_hub import InferenceClient
except ModuleNotFoundError as e:
    raise RuntimeError(
        "Missing package: huggingface_hub. Install once with: pip install huggingface_hub"
    ) from e

try:
    from sklearn.metrics import accuracy_score, f1_score
except ModuleNotFoundError as e:
    raise RuntimeError(
        "Missing package: scikit-learn. Install once with: pip install scikit-learn"
    ) from e

print("Imports OK")

Imports OK


In [ ]:
# Credentials + experiment config
HF_TOKEN = os.getenv("HF_TOKEN", "")
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient

        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        HF_TOKEN = ""

if not HF_TOKEN:
    raise ValueError("HF_TOKEN not found. Set env var or Kaggle secret named HF_TOKEN.")

SEED = 42
N_SAMPLES_PER_DATASET = 50
BATCH_SIZE = 5  # User-requested default
MAX_TOKENS = 500
TEMPERATURE = 0.0
MAX_402_RETRIES = 8
RETRY_SLEEP_SECONDS = 30.0

DATASETS = [
    {"name": "tweet_sentiment", "path": "tweet_eval", "subset": "sentiment", "split": "test"},
    {
        "name": "social_bias_frames",
        "source": "sbic_archive",
        "url": "https://homes.cs.washington.edu/~msap/social-bias-frames/SBIC.v2.tgz",
        "split_file": "SBIC.v2.agg.tst.csv",
        "text_col": "post",
        "label_col": "hasBiasedImplication",
    },
    # lex_glue/scotus temporarily disabled by request
    {"name": "ag_news", "path": "ag_news", "subset": None, "split": "test"},
]

MODEL_SPECS = [
    {
        "name": "qwen_1_5b",
        "model_candidates": ["Qwen/Qwen2.5-1.5B-Instruct"],
    },
    {
        "name": "llama_8b",
        "model_candidates": ["meta-llama/Meta-Llama-3-8B-Instruct"],
    },
    {
        "name": "gpt_oss_20b",
        "model_candidates": ["openai/gpt-oss-20b"],
    },
    {
        "name": "mixtral_8x22b",
        "model_candidates": ["mistralai/Mixtral-8x22B-Instruct-v0.1"],
    },
]

client = InferenceClient(token=HF_TOKEN)
rng = np.random.default_rng(SEED)

print("Config loaded")

Config loaded


In [6]:
def normalize_label_text(s: str) -> str:
    s = str(s).strip().lower()
    s = re.sub(r"[^a-z0-9]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def detect_text_column(df: pd.DataFrame) -> str:
    candidates = ["text", "sentence", "content", "document", "review", "question", "premise"]
    for c in candidates:
        if c in df.columns:
            return c

    object_cols = [c for c in df.columns if df[c].dtype == object]
    if object_cols:
        return object_cols[0]

    raise ValueError(f"Could not detect a text column in columns={list(df.columns)}")


@lru_cache(maxsize=1)
def download_social_bias_frames_archive(url: str) -> bytes:
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    return response.content


def load_social_bias_frames_frame(ds_cfg):
    archive_bytes = download_social_bias_frames_archive(ds_cfg["url"])
    with tarfile.open(fileobj=io.BytesIO(archive_bytes), mode="r:gz") as tf:
        member = tf.extractfile(ds_cfg["split_file"])
        if member is None:
            raise ValueError(f"Could not find {ds_cfg['split_file']} in social_bias_frames archive")
        df = pd.read_csv(member)

    keep_cols = [c for c in df.columns if str(c).strip() and not str(c).startswith("Unnamed")]
    df = df[keep_cols].copy()
    df["label_id"] = pd.to_numeric(df[ds_cfg["label_col"]], errors="coerce").fillna(0).astype(int)
    df["label_name"] = df["label_id"].map({0: "not_biased", 1: "biased"})
    df["text"] = df[ds_cfg["text_col"]].astype(str)

    sampled = stratified_sample_df(df[["text", "label_id", "label_name"]], "label_id", N_SAMPLES_PER_DATASET, SEED)
    sampled["sample_id"] = [f"{ds_cfg['name']}_{i:03d}" for i in range(len(sampled))]
    return sampled[["sample_id", "text", "label_id", "label_name"]], ["not_biased", "biased"]


def stratified_sample_df(df: pd.DataFrame, label_col: str, n: int, seed: int) -> pd.DataFrame:
    if label_col not in df.columns:
        return df.sample(n=min(n, len(df)), random_state=seed).reset_index(drop=True)

    counts = df[label_col].value_counts(dropna=False)
    if counts.empty:
        return df.sample(n=min(n, len(df)), random_state=seed).reset_index(drop=True)

    proportions = counts / counts.sum()
    alloc = (proportions * n).astype(int)

    # Keep representation from all classes when possible
    for cls in counts.index:
        if alloc.loc[cls] == 0 and counts.loc[cls] > 0 and alloc.sum() < n:
            alloc.loc[cls] = 1

    remainder = n - int(alloc.sum())
    if remainder > 0:
        fractions = (proportions * n) - (proportions * n).astype(int)
        for cls in fractions.sort_values(ascending=False).index:
            if remainder == 0:
                break
            if alloc.loc[cls] < counts.loc[cls]:
                alloc.loc[cls] += 1
                remainder -= 1

    sampled_parts = []
    for cls, k in alloc.items():
        k = int(min(k, counts.loc[cls]))
        if k <= 0:
            continue
        part = df[df[label_col] == cls].sample(n=k, random_state=seed)
        sampled_parts.append(part)

    sampled = pd.concat(sampled_parts, axis=0) if sampled_parts else df.head(0)

    if len(sampled) < min(n, len(df)):
        missing = min(n, len(df)) - len(sampled)
        remaining = df.loc[~df.index.isin(sampled.index)]
        if len(remaining) > 0:
            sampled = pd.concat([sampled, remaining.sample(n=min(missing, len(remaining)), random_state=seed)], axis=0)

    sampled = sampled.sample(frac=1, random_state=seed).reset_index(drop=True)
    return sampled


def chunk_df(df: pd.DataFrame, batch_size: int):
    for i in range(0, len(df), batch_size):
        yield df.iloc[i : i + batch_size].copy()


def to_builtin(obj):
    if obj is None or isinstance(obj, (str, int, float, bool)):
        return obj
    if isinstance(obj, dict):
        return {k: to_builtin(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_builtin(v) for v in obj]
    if hasattr(obj, "model_dump"):
        return to_builtin(obj.model_dump())
    if hasattr(obj, "__dict__"):
        return {k: to_builtin(v) for k, v in vars(obj).items()}
    return str(obj)


def summarize_logprobs(choice_obj):
    logprobs_obj = getattr(choice_obj, "logprobs", None)
    if logprobs_obj is None:
        return {"logprobs_available": False, "token_logprobs": [], "mean_logprob": None, "min_logprob": None, "max_logprob": None}

    raw = to_builtin(logprobs_obj)

    token_lps = []
    if isinstance(raw, dict):
        content = raw.get("content", None)
        if isinstance(content, list):
            for item in content:
                if isinstance(item, dict) and item.get("logprob") is not None:
                    try:
                        token_lps.append(float(item["logprob"]))
                    except Exception:
                        pass

    mean_lp = float(np.mean(token_lps)) if token_lps else None
    min_lp = float(np.min(token_lps)) if token_lps else None
    max_lp = float(np.max(token_lps)) if token_lps else None

    return {
        "logprobs_available": bool(token_lps),
        "token_logprobs": token_lps,
        "mean_logprob": mean_lp,
        "min_logprob": min_lp,
        "max_logprob": max_lp,
        "raw_logprobs": raw,
    }


def extract_json_array(text: str):
    s = text.strip()
    s = re.sub(r"^```(?:json)?", "", s).strip()
    s = re.sub(r"```$", "", s).strip()

    try:
        parsed = json.loads(s)
        if isinstance(parsed, list):
            return parsed
        if isinstance(parsed, dict):
            if "predictions" in parsed and isinstance(parsed["predictions"], list):
                return parsed["predictions"]
            if "items" in parsed and isinstance(parsed["items"], list):
                return parsed["items"]
    except Exception:
        pass

    m = re.search(r"\[.*\]", s, flags=re.DOTALL)
    if m:
        candidate = m.group(0)
        try:
            parsed = json.loads(candidate)
            if isinstance(parsed, list):
                return parsed
        except Exception:
            return None

    return None


def normalize_pred_to_allowed(pred_label: str, allowed_labels):
    if pred_label is None:
        return None

    allowed_norm = {normalize_label_text(lbl): lbl for lbl in allowed_labels}
    p = normalize_label_text(pred_label)
    if p in allowed_norm:
        return allowed_norm[p]

    for key, original in allowed_norm.items():
        if p == key or p in key or key in p:
            return original

    if {"biased", "not biased"}.issubset(set(allowed_norm.keys())):
        biased_label = allowed_norm["biased"]
        not_biased_label = allowed_norm["not biased"]
        positive = {"yes", "true", "1", "biased", "offensive", "harmful", "positive"}
        negative = {"no", "false", "0", "not biased", "unbiased", "negative", "benign"}
        if p in positive:
            return biased_label
        if p in negative:
            return not_biased_label

    return None


print("Helpers ready")

Helpers ready


In [7]:
def load_dataset_frame(ds_cfg):
    if ds_cfg.get("source") == "sbic_archive":
        return load_social_bias_frames_frame(ds_cfg)

    if ds_cfg["subset"] is None:
        ds = load_dataset(ds_cfg["path"], split=ds_cfg["split"])
    else:
        ds = load_dataset(ds_cfg["path"], ds_cfg["subset"], split=ds_cfg["split"])

    df = ds.to_pandas()
    text_col = detect_text_column(df)

    if "label" not in df.columns:
        raise ValueError(f"Dataset {ds_cfg['name']} has no 'label' column; got columns={list(df.columns)}")

    label_names = None
    try:
        label_names = ds.features["label"].names
    except Exception:
        label_names = None

    if label_names is None:
        unique_labels = sorted(df["label"].dropna().unique().tolist())
        label_names = [str(x) for x in unique_labels]

    id_to_name = {i: name for i, name in enumerate(label_names)}

    df = df[[text_col, "label"]].copy()
    df = df.rename(columns={text_col: "text", "label": "label_id"})
    df["label_id"] = df["label_id"].astype(int)
    df["label_name"] = df["label_id"].map(id_to_name).fillna(df["label_id"].astype(str))

    sampled = stratified_sample_df(df, "label_id", N_SAMPLES_PER_DATASET, SEED)
    sampled["sample_id"] = [f"{ds_cfg['name']}_{i:03d}" for i in range(len(sampled))]

    return sampled[["sample_id", "text", "label_id", "label_name"]], label_names


def few_shot_blocks(dataset_name, label_names):
    if dataset_name == "tweet_sentiment":
        return [
            {"text": "I waited 2 hours and still no support. Totally disappointed.", "label": "negative"},
            {"text": "Service was okay, nothing special, nothing terrible.", "label": "neutral"},
            {"text": "Absolutely loved the update, everything is smoother now!", "label": "positive"},
        ]

    if dataset_name == "social_bias_frames":
        return [
            {"text": "Women are usually bad at leadership.", "label": "biased"},
            {"text": "The article describes a weather report.", "label": "not_biased"},
            {"text": "That group is lazy and should be avoided.", "label": "biased"},
        ]

    if dataset_name == "ag_news":
        return [
            {"text": "The central bank signaled a rate pause as inflation cooled in Q2.", "label": "Business"},
            {"text": "The team secured the championship after a dramatic overtime finish.", "label": "Sports"},
            {"text": "Researchers released a new battery chemistry for longer EV range.", "label": "Sci/Tech"},
            {"text": "Leaders met in Geneva to discuss sanctions and a ceasefire proposal.", "label": "World"},
        ]

    if dataset_name == "lex_scotus":
        # These are manual legal-domain examples; labels are normalized to available labels at runtime.
        return [
            {"text": "The defendant contests the admissibility of a confession obtained during custodial interrogation.", "label": "Criminal Procedure"},
            {"text": "The plaintiff alleges discrimination by a state employer under equal protection principles.", "label": "Civil Rights"},
            {"text": "A dispute over patent validity and infringement reached the federal circuit.", "label": "Economic Activity"},
        ]

    return []


def adapt_few_shots_to_allowed(few_shots, allowed_labels):
    adapted = []
    for ex in few_shots:
        mapped = normalize_pred_to_allowed(ex["label"], allowed_labels)
        if mapped is None and len(allowed_labels) > 0:
            mapped = allowed_labels[0]
        adapted.append({"text": ex["text"], "label": mapped})
    return adapted


def build_messages(dataset_name, batch_df, allowed_labels):
    few_shots = adapt_few_shots_to_allowed(few_shot_blocks(dataset_name, allowed_labels), allowed_labels)

    label_line = ", ".join(allowed_labels)

    fs_lines = []
    for i, ex in enumerate(few_shots, start=1):
        fs_lines.append(f"FEW_SHOT_{i}: text={json.dumps(ex['text'])} -> label={json.dumps(ex['label'])}")

    items = []
    for _, row in batch_df.iterrows():
        items.append({"id": row["sample_id"], "text": row["text"]})

    user_prompt = (
        f"Dataset: {dataset_name}\n"
        "Task: Multi-class text classification.\n"
        f"Allowed labels (exact strings): [{label_line}]\n\n"
        + "Few-shot examples:\n"
        + "\n".join(fs_lines)
        + "\n\nNow classify ALL items below.\n"
        + "Return ONLY valid JSON array (no markdown), with one object per item:\n"
        + "[{\"id\": \"...\", \"label\": \"...\", \"reason\": \"short rationale\"}]\n"
        + "Use only labels from allowed labels list.\n"
        + "Items:\n"
        + json.dumps(items, ensure_ascii=False)
    )

    messages = [
        {
            "role": "system",
            "content": "You are a careful classifier. Follow instructions exactly and return strict JSON only.",
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]
    return messages


print("Dataset + prompt utilities ready")

Dataset + prompt utilities ready


In [8]:
resolved_model_ids = {}
logprobs_unsupported_models = set()


def is_402_error_message(message: str) -> bool:
    msg = str(message).lower()
    return "402" in msg and ("payment required" in msg or "depleted" in msg)


def is_logprobs_unsupported_message(message: str) -> bool:
    msg = str(message).lower()
    return "logprobs" in msg and "not supported" in msg


def call_chat_completion_with_retry(
    model_id,
    messages,
    max_tokens,
    temperature,
    with_logprobs,
    max_402_retries,
    retry_sleep_seconds,
):
    stage = "with_logprobs" if with_logprobs else "without_logprobs"
    attempt_errors = []

    for attempt in range(max_402_retries + 1):
        try:
            kwargs = {
                "model": model_id,
                "messages": messages,
                "max_tokens": max_tokens,
                "temperature": temperature,
            }
            if with_logprobs:
                kwargs["logprobs"] = True
                kwargs["top_logprobs"] = 5

            response = client.chat_completion(**kwargs)
            return response, attempt_errors
        except Exception as e:
            err_text = str(e)
            attempt_errors.append(
                {
                    "model_id": model_id,
                    "stage": stage,
                    "attempt": attempt + 1,
                    "error": err_text,
                }
            )

            if is_402_error_message(err_text) and attempt < max_402_retries:
                time.sleep(retry_sleep_seconds)
                continue
            break

    return None, attempt_errors


def chat_with_fallback(model_name, model_candidates, messages, max_tokens, temperature):
    errors = []

    # Reuse successful resolved model ID to avoid repeated fallback probing
    if model_name in resolved_model_ids:
        model_candidates = [resolved_model_ids[model_name]] + [m for m in model_candidates if m != resolved_model_ids[model_name]]

    for model_id in model_candidates:
        can_try_logprobs = model_name not in logprobs_unsupported_models

        if can_try_logprobs:
            response, retry_errors = call_chat_completion_with_retry(
                model_id=model_id,
                messages=messages,
                max_tokens=max_tokens,
                temperature=temperature,
                with_logprobs=True,
                max_402_retries=MAX_402_RETRIES,
                retry_sleep_seconds=RETRY_SLEEP_SECONDS,
            )
            errors.extend(retry_errors)

            if response is not None:
                resolved_model_ids[model_name] = model_id
                return response, model_id, True, errors

            if any(is_logprobs_unsupported_message(e["error"]) for e in retry_errors):
                logprobs_unsupported_models.add(model_name)

        response, retry_errors = call_chat_completion_with_retry(
            model_id=model_id,
            messages=messages,
            max_tokens=max_tokens,
            temperature=temperature,
            with_logprobs=False,
            max_402_retries=MAX_402_RETRIES,
            retry_sleep_seconds=RETRY_SLEEP_SECONDS,
        )
        errors.extend(retry_errors)

        if response is not None:
            resolved_model_ids[model_name] = model_id
            return response, model_id, False, errors

    return None, None, False, errors


def classify_batch(dataset_name, batch_df, allowed_labels, model_spec):
    messages = build_messages(dataset_name, batch_df, allowed_labels)

    response, used_model_id, used_logprobs_request, errors = chat_with_fallback(
        model_name=model_spec["name"],
        model_candidates=model_spec["model_candidates"],
        messages=messages,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
    )

    if response is None:
        return {
            "ok": False,
            "used_model_id": None,
            "used_logprobs_request": False,
            "logprobs_summary": {"logprobs_available": False, "mean_logprob": None, "min_logprob": None, "max_logprob": None, "token_logprobs": []},
            "raw_text": None,
            "pred_map": {},
            "errors": errors,
        }

    choice0 = response.choices[0]
    raw_text = choice0.message.content
    parsed = extract_json_array(raw_text)

    pred_map = {}
    if isinstance(parsed, list):
        for item in parsed:
            if not isinstance(item, dict):
                continue
            sid = item.get("id")
            pred = item.get("label")
            pred_norm = normalize_pred_to_allowed(pred, allowed_labels)
            if sid is not None:
                pred_map[str(sid)] = pred_norm

    lp_summary = summarize_logprobs(choice0)

    return {
        "ok": True,
        "used_model_id": used_model_id,
        "used_logprobs_request": used_logprobs_request,
        "logprobs_summary": lp_summary,
        "raw_text": raw_text,
        "pred_map": pred_map,
        "errors": errors,
    }


print("Inference helpers ready")

Inference helpers ready


In [9]:
all_prediction_rows = []
run_errors = []

dataset_samples = {}
dataset_allowed_labels = {}

for ds_cfg in DATASETS:
    ds_name = ds_cfg["name"]
    print(f"Loading dataset: {ds_name}")
    sampled_df, label_names = load_dataset_frame(ds_cfg)
    dataset_samples[ds_name] = sampled_df
    dataset_allowed_labels[ds_name] = label_names
    print(f"  sampled={len(sampled_df)} classes={len(label_names)}")

for ds_cfg in DATASETS:
    ds_name = ds_cfg["name"]
    sampled_df = dataset_samples[ds_name]
    allowed_labels = dataset_allowed_labels[ds_name]

    print(f"\n=== Dataset: {ds_name} ===")

    # Round-robin models per batch to avoid hammering one model continuously.
    batches = list(chunk_df(sampled_df, BATCH_SIZE))
    for batch_no, batch_df in enumerate(batches, start=1):
        print(f"Batch {batch_no}/{len(batches)}")

        for model_spec in MODEL_SPECS:
            model_name = model_spec["name"]
            print(f"  Running model={model_name} on {ds_name} batch={batch_no}")

            result = classify_batch(ds_name, batch_df, allowed_labels, model_spec)

            batch_lp = result["logprobs_summary"]
            batch_errors = result["errors"]
            if batch_errors:
                for err in batch_errors:
                    run_errors.append(
                        {
                            "dataset": ds_name,
                            "model": model_name,
                            "batch_no": batch_no,
                            **err,
                        }
                    )

            for _, row in batch_df.iterrows():
                sid = str(row["sample_id"])
                pred = result["pred_map"].get(sid, None)

                all_prediction_rows.append(
                    {
                        "dataset": ds_name,
                        "sample_id": sid,
                        "text": row["text"],
                        "true_label_id": int(row["label_id"]),
                        "true_label": row["label_name"],
                        "model": model_name,
                        "model_id_used": result["used_model_id"],
                        "pred_label": pred,
                        "correct": (pred == row["label_name"]) if pred is not None else False,
                        "batch_no": batch_no,
                        "batch_size": int(len(batch_df)),
                        "used_logprobs_request": bool(result["used_logprobs_request"]),
                        "logprobs_available": bool(batch_lp.get("logprobs_available", False)),
                        "mean_logprob": batch_lp.get("mean_logprob"),
                        "min_logprob": batch_lp.get("min_logprob"),
                        "max_logprob": batch_lp.get("max_logprob"),
                        "token_logprobs": json.dumps(batch_lp.get("token_logprobs", []), ensure_ascii=False),
                        "raw_response_text": result["raw_text"],
                        "had_errors": len(batch_errors) > 0,
                        "errors_json": json.dumps(batch_errors, ensure_ascii=False),
                    }
                )

pred_df = pd.DataFrame(all_prediction_rows)
errors_df = pd.DataFrame(run_errors)

print(f"\nCollected rows: {len(pred_df)}")
if len(errors_df) > 0:
    print(f"Logged warnings/errors: {len(errors_df)}")
else:
    print("No logged inference warnings/errors.")

Loading dataset: tweet_sentiment


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sentiment/train-00000-of-00001.parquet:   0%|          | 0.00/3.78M [00:00<?, ?B/s]

sentiment/test-00000-of-00001.parquet:   0%|          | 0.00/901k [00:00<?, ?B/s]

...

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

  sampled=50 classes=4

=== Dataset: tweet_sentiment ===
Batch 1/10
  Running model=qwen_1_5b on tweet_sentiment batch=1
  Running model=llama_8b on tweet_sentiment batch=1
  Running model=gpt_oss_20b on tweet_sentiment batch=1
  Running model=mixtral_8x22b on tweet_sentiment batch=1
Batch 2/10
  Running model=qwen_1_5b on tweet_sentiment batch=2
  Running model=llama_8b on tweet_sentiment batch=2
  Running model=gpt_oss_20b on tweet_sentiment batch=2
  Running model=mixtral_8x22b on tweet_sentiment batch=2
Batch 3/10
  Running model=qwen_1_5b on tweet_sentiment batch=3
  Running model=llama_8b on tweet_sentiment batch=3
  Running model=gpt_oss_20b on tweet_sentiment batch=3
  Running model=mixtral_8x22b on tweet_sentiment batch=3
Batch 4/10
  Running model=qwen_1_5b on tweet_sentiment batch=4
  Running model=llama_8b on tweet_sentiment batch=4
  Running model=gpt_oss_20b on tweet_sentiment batch=4
  Running model=mixtral_8x22b on tweet_sentiment batch=4
Batch 5/10
  Running model=qwen

In [10]:
if pred_df.empty:
    raise RuntimeError("No predictions collected. Check token/model access and rerun.")

# Wide table with one row per exact sample and per-model prediction columns
base_cols = ["dataset", "sample_id", "text", "true_label_id", "true_label"]
base_df = pred_df[base_cols].drop_duplicates().copy()

wide_preds = pred_df.pivot_table(
    index=["dataset", "sample_id"],
    columns="model",
    values="pred_label",
    aggfunc="first",
).reset_index()

wide_df = base_df.merge(wide_preds, on=["dataset", "sample_id"], how="left")

model_names = [m["name"] for m in MODEL_SPECS]

def compute_major_vote(row):
    preds = [row.get(m) for m in model_names if pd.notna(row.get(m)) and row.get(m) is not None]
    if not preds:
        return None
    c = Counter(preds)
    top = c.most_common()
    if len(top) > 1 and top[0][1] == top[1][1]:
        return "TIE"
    return top[0][0]


def compute_all_agree(row):
    preds = [row.get(m) for m in model_names if pd.notna(row.get(m)) and row.get(m) is not None]
    if len(preds) != len(model_names):
        return False
    return len(set(preds)) == 1


wide_df["all_agree"] = wide_df.apply(compute_all_agree, axis=1)
wide_df["major_vote"] = wide_df.apply(compute_major_vote, axis=1)
wide_df["major_vote_correct"] = wide_df["major_vote"] == wide_df["true_label"]

# Metrics per model and dataset
metric_rows = []
for ds_name, ds_part in wide_df.groupby("dataset"):
    y_true = ds_part["true_label"].tolist()

    for m in model_names:
        y_pred = ds_part[m].tolist() if m in ds_part.columns else [None] * len(ds_part)
        valid_mask = [pd.notna(p) for p in y_pred]
        y_true_valid = [t for t, ok in zip(y_true, valid_mask) if ok]
        y_pred_valid = [p for p in y_pred if pd.notna(p)]

        if len(y_pred_valid) == 0:
            acc = np.nan
            macro_f1 = np.nan
            weighted_f1 = np.nan
            coverage = 0.0
        else:
            acc = accuracy_score(y_true_valid, y_pred_valid)
            macro_f1 = f1_score(y_true_valid, y_pred_valid, average="macro", zero_division=0)
            weighted_f1 = f1_score(y_true_valid, y_pred_valid, average="weighted", zero_division=0)
            coverage = len(y_pred_valid) / len(y_true)

        metric_rows.append({
            "dataset": ds_name,
            "model": m,
            "n_samples": len(y_true),
            "coverage": coverage,
            "accuracy": acc,
            "macro_f1": macro_f1,
            "weighted_f1": weighted_f1,
            "all_agree_rate": float(ds_part["all_agree"].mean()),
            "major_vote_accuracy": float(ds_part["major_vote_correct"].mean()),
        })

    mv = ds_part["major_vote"].tolist()
    mv_valid_mask = [pd.notna(p) and p != "TIE" for p in mv]
    y_true_mv = [t for t, ok in zip(y_true, mv_valid_mask) if ok]
    y_pred_mv = [p for p in mv if pd.notna(p) and p != "TIE"]

    if len(y_pred_mv) == 0:
        mv_acc = np.nan
        mv_macro = np.nan
        mv_weighted = np.nan
        mv_cov = 0.0
    else:
        mv_acc = accuracy_score(y_true_mv, y_pred_mv)
        mv_macro = f1_score(y_true_mv, y_pred_mv, average="macro", zero_division=0)
        mv_weighted = f1_score(y_true_mv, y_pred_mv, average="weighted", zero_division=0)
        mv_cov = len(y_pred_mv) / len(y_true)

    metric_rows.append({
        "dataset": ds_name,
        "model": "major_vote",
        "n_samples": len(y_true),
        "coverage": mv_cov,
        "accuracy": mv_acc,
        "macro_f1": mv_macro,
        "weighted_f1": mv_weighted,
        "all_agree_rate": float(ds_part["all_agree"].mean()),
        "major_vote_accuracy": float(ds_part["major_vote_correct"].mean()),
    })

metrics_df = pd.DataFrame(metric_rows)

print("Metrics table ready")
display(metrics_df.sort_values(["dataset", "model"]).reset_index(drop=True))

Metrics table ready


,dataset,model,n_samples,coverage,accuracy,macro_f1,weighted_f1,all_agree_rate,major_vote_accuracy
0,ag_news,gpt_oss_20b,50,0.5,0.760000,0.737374,0.727677,0.0,0.46
1,ag_news,llama_8b,50,0.1,0.800000,0.666667,0.733333,0.0,0.46
2,ag_news,major_vote,50,0.6,0.766667,0.740476,0.729841,0.0,0.46
3,ag_news,mixtral_8x22b,50,0.0,NaN,NaN,NaN,0.0,0.46
4,ag_news,qwen_1_5b,50,0.0,NaN,NaN,NaN,0.0,0.46
5,social_bias_frames,gpt_oss_20b,50,0.1,0.000000,0.000000,0.000000,0.0,0.20
6,social_bias_frames,llama_8b,50,0.5,0.400000,0.363328,0.406112,0.0,0.20
7,social_bias_frames,major_vote,50,0.5,0.400000,0.363328,0.406112,0.0,0.20
8,social_bias_frames,mixtral_8x22b,50,0.0,NaN,NaN,NaN,0.0,0.20
9,social_bias_frames,qwen_1_5b,50,0.0,NaN,NaN,NaN,0.0,0.20


In [11]:
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out_dir = Path("results/hf_llms_comparison")
out_dir.mkdir(parents=True, exist_ok=True)

pred_path_csv = out_dir / f"predictions_long_{ts}.csv"
pred_path_jsonl = out_dir / f"predictions_long_{ts}.jsonl"
wide_path_csv = out_dir / f"predictions_wide_{ts}.csv"
metrics_path_csv = out_dir / f"metrics_{ts}.csv"
errors_path_csv = out_dir / f"errors_{ts}.csv"
summary_path_json = out_dir / f"summary_{ts}.json"

pred_df.to_csv(pred_path_csv, index=False)
pred_df.to_json(pred_path_jsonl, orient="records", lines=True, force_ascii=False)
wide_df.to_csv(wide_path_csv, index=False)
metrics_df.to_csv(metrics_path_csv, index=False)

if len(errors_df) > 0:
    errors_df.to_csv(errors_path_csv, index=False)

summary = {
    "timestamp": ts,
    "n_prediction_rows": int(len(pred_df)),
    "n_unique_samples": int(wide_df[["dataset", "sample_id"]].drop_duplicates().shape[0]),
    "datasets": sorted(wide_df["dataset"].unique().tolist()),
    "models": model_names,
    "resolved_model_ids": resolved_model_ids,
    "logprobs_available_rate": float(pred_df["logprobs_available"].mean()),
    "files": {
        "predictions_long_csv": str(pred_path_csv),
        "predictions_long_jsonl": str(pred_path_jsonl),
        "predictions_wide_csv": str(wide_path_csv),
        "metrics_csv": str(metrics_path_csv),
        "errors_csv": str(errors_path_csv) if len(errors_df) > 0 else None,
    },
}

with open(summary_path_json, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("Saved artifacts:")
print(f"- {pred_path_csv}")
print(f"- {pred_path_jsonl}")
print(f"- {wide_path_csv}")
print(f"- {metrics_path_csv}")
if len(errors_df) > 0:
    print(f"- {errors_path_csv}")
print(f"- {summary_path_json}")

Saved artifacts:
- results/hf_llms_comparison/predictions_long_20260425_080936.csv
- results/hf_llms_comparison/predictions_long_20260425_080936.jsonl
- results/hf_llms_comparison/predictions_wide_20260425_080936.csv
- results/hf_llms_comparison/metrics_20260425_080936.csv
- results/hf_llms_comparison/errors_20260425_080936.csv
- results/hf_llms_comparison/summary_20260425_080936.json


In [12]:
print("Sample-level view (exact samples + per-model predictions):")
display(wide_df.head(20))

if len(errors_df) > 0:
    print("\nDead ends / endpoint issues encountered (first 20):")
    display(errors_df.head(20))
else:
    print("\nNo endpoint dead ends captured in this run.")

Sample-level view (exact samples + per-model predictions):


,dataset,sample_id,text,true_label_id,true_label,gpt_oss_20b,llama_8b,all_agree,major_vote,major_vote_correct
0,tweet_sentiment,tweet_sentiment_000,Bannon bringing Reagan back! #magahttps://t.co...,1,neutral,NaN,positive,False,positive,False
1,tweet_sentiment,tweet_sentiment_001,"FrPavone: ""total number of deaths by capital p...",0,negative,NaN,neutral,False,neutral,False
2,tweet_sentiment,tweet_sentiment_002,The Socialist/Fascist Party (aka. Dems) is dri...,0,negative,NaN,negative,False,negative,True
3,tweet_sentiment,tweet_sentiment_003,"Gold heart earrings, Textured gold heart charm...",2,positive,NaN,neutral,False,neutral,False
4,tweet_sentiment,tweet_sentiment_004,Here's Putin's man in our White House.@Lindsey...,1,neutral,NaN,negative,False,negative,False
5,tweet_sentiment,tweet_sentiment_005,"@user @user When Hillary said, ""Basket of Depl...",2,positive,positive,positive,False,positive,True
6,tweet_sentiment,tweet_sentiment_006,Racists masochists fascist all the hate words ...,0,negative,negative,negative,False,negative,True
7,tweet_sentiment,tweet_sentiment_007,#BreakingNewslist of the terrorist groups at #...,0,negative,neutral,neutral,False,neutral,False
8,tweet_sentiment,tweet_sentiment_008,"yesterday was #NationalFastFoodDay, a day wher...",0,negative,negative,negative,False,negative,True
9,tweet_sentiment,tweet_sentiment_009,"Published on Nov 18, 2016Speech by Stephen K. ...",1,neutral,neutral,neutral,False,neutral,True



Dead ends / endpoint issues encountered (first 20):


,dataset,model,batch_no,model_id,stage,attempt,error
0,tweet_sentiment,qwen_1_5b,1,Qwen/Qwen2.5-1.5B-Instruct,with_logprobs,1,(Request ID: Root=1-69ec71ed-79d39388617d82684...
1,tweet_sentiment,qwen_1_5b,1,Qwen/Qwen2.5-1.5B-Instruct,without_logprobs,1,(Request ID: Root=1-69ec71ed-424130d979d6320b5...
2,tweet_sentiment,gpt_oss_20b,1,openai/gpt-oss-20b,with_logprobs,1,(Request ID: req_01kq1st4kye29vsfczwaj84ykb)\n...
3,tweet_sentiment,mixtral_8x22b,1,mistralai/Mixtral-8x22B-Instruct-v0.1,with_logprobs,1,(Request ID: Root=1-69ec71f1-702f619f78231a761...
4,tweet_sentiment,mixtral_8x22b,1,mistralai/Mixtral-8x22B-Instruct-v0.1,without_logprobs,1,(Request ID: Root=1-69ec71f1-4896f6da424b4bd00...
5,tweet_sentiment,qwen_1_5b,2,Qwen/Qwen2.5-1.5B-Instruct,with_logprobs,1,(Request ID: Root=1-69ec71f1-7e7049d7611099aa6...
6,tweet_sentiment,qwen_1_5b,2,Qwen/Qwen2.5-1.5B-Instruct,without_logprobs,1,(Request ID: Root=1-69ec71f1-3bd7121b6d00ffa92...
7,tweet_sentiment,mixtral_8x22b,2,mistralai/Mixtral-8x22B-Instruct-v0.1,with_logprobs,1,(Request ID: Root=1-69ec71f5-704f04e761ec169f7...
8,tweet_sentiment,mixtral_8x22b,2,mistralai/Mixtral-8x22B-Instruct-v0.1,without_logprobs,1,(Request ID: Root=1-69ec71f5-052c27001d487bcf6...
9,tweet_sentiment,qwen_1_5b,3,Qwen/Qwen2.5-1.5B-Instruct,with_logprobs,1,(Request ID: Root=1-69ec71f5-3c187c3b1e31f0066...


In [1]:
import os
import time
from huggingface_hub import InferenceClient, AsyncInferenceClient

from huggingface_hub import InferenceClient, InferenceTimeoutError
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

import dotenv
dotenv.load_dotenv()

# Define exactly which errors are "retry-worthy"
retryable_errors = (InferenceTimeoutError, ConnectionError)

# 1. Credential Injection
# It is best practice to set your token as an environment variable:
# Linux/Mac: export HF_TOKEN="your_hf_token_here"
# Windows: setx HF_TOKEN "your_hf_token_here"
hf_token = os.getenv("HF_TOKEN")

if not hf_token:
    print("WARNING: HF_TOKEN not found in environment. Please paste it below:")
    hf_token = input("Token: ").strip()

# Initialize the client
client = AsyncInferenceClient(token=hf_token)

# 2. Define the Target Architectures
# We are targeting the specific Instruct variants for Q&A formatting
PHI4_MODEL_ID = "microsoft/Phi-4-mini-instruct"
# We are targeting a smaller hosted model plus a larger one for comparison
QWEN_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
LLAMA_MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"
GPTOSS_MODEL_ID = "openai/gpt-oss-20b"
MIXRAL_MODEL_ID = "mistralai/Mixtral-8x22B-Instruct-v0.1"

# 3. Define the System and User Prompts
messages = [
    {"role": "system", "content": "You are a highly analytical AI assistant. Be concise."},
    {"role": "user", "content": "Explain the concept of 'Zero-Shot Learning' in one paragraph."}
]

def is_retryable_402(exception):
    """Returns True if error is a HfHubHTTPError with status code 402."""
    return exception.response.status_code == 402



async def query_model(model_id, chat_messages):
    print(f"\n[{model_id}] Initializing connection...")
    try:
        # The chat_completion endpoint automatically formats the prompt 
        # to match the specific model's required chat template (e.g., <|user|>, [INST]).
        response = await client.chat_completion(
            model=model_id,
            messages=chat_messages,
            max_tokens=150,
            temperature=0.1, # Low temperature for analytical consistency
        )
        
        # Extract the actual text from the response payload
        output = response.choices[0].message.content
        print(f"SUCCESS. Output:\n{output}\n")
        print("-" * 50)
        
    except Exception as e:
        print(f"FAILED. Error details: {e}\n")
        print("-" * 50)

for i in range(10):
    print(i)
    print("Starting Model Inference Sequence...\n")
    
    # Test 1: Phi-4 Mini (4.0B Parameters)
    # This is a lightweight model and should run almost instantly on the free tier.
    # await query_model(PHI4_MODEL_ID, messages)
    
    # Test 2: Qwen 2.5 1.5B Instruct
    # This is a smaller hosted model with a working provider mapping.
    # await query_model(QWEN_MODEL_ID, messages)
    
    # Test 3: LLaMA 3 8B Instruct
    # This is a larger model that may trigger a cold start, but it is still within
    # the free tier limits and should be a good test of the system's handling of larger models.
    # query_model(LLAMA_MODEL_ID, messages)
    
    # # Test 4: GPT-OSS 20B
    # query_model(GPTOSS_MODEL_ID, messages)
    
    # # Test 5: Mixtral 8x22B (141B Total Parameters, 39B Active)
    # # WARNING: See operational briefing below regarding this model.
    await query_model(MIXRAL_MODEL_ID, messages)
    
    time.sleep(10)  # Brief pause between queries to avoid rate limits
    

0
Starting Model Inference Sequence...


[mistralai/Mixtral-8x22B-Instruct-v0.1] Initializing connection...
SUCCESS. Output:
 Zero-Shot Learning is a machine learning method where a model learns to recognize or classify objects, concepts, or attributes it has never encountered during training. This is achieved by leveraging additional information about the categories, such as attributes or descriptions, allowing the model to generalize and make predictions about unseen classes based on their relationships with seen classes. This approach is particularly useful when labeled data is scarce or unavailable for certain categories.

--------------------------------------------------
1
Starting Model Inference Sequence...


[mistralai/Mixtral-8x22B-Instruct-v0.1] Initializing connection...


CancelledError: 

In [23]:
res_path = pathlib.Path().resolve() / "results"

In [25]:
import shutil
import base64
from IPython.display import HTML

# Zip it first
shutil.make_archive('my_data', 'zip', root_dir=res_path)

# Create a link with a download attribute
def create_download_link(filename):
    with open(filename, 'rb') as f:
        data = f.read()
    b64 = base64.b64encode(data).decode()
    return f'<a href="data:application/zip;base64,{b64}" download="{filename}">Click here to download {filename}</a>'

HTML(create_download_link('my_data.zip'))
